# Подбор гиперпараметров

Полный pipeline передаётся в GridSearchCV, поэтому статистики обучаются внутри каждого train-фолда. Test не участвует в подборе.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Подключаем модули проекта при запуске из папки notebooks.
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "config.ini").exists():
    raise RuntimeError("Запустите ноутбук из папки проекта или notebooks")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import DataPreprocessor
from src.pipelines import build_pipeline
from src.train import metrics



## Подготовка данных и пространства поиска


In [ ]:
processor = DataPreprocessor()
df = processor.load_data()

X_train, X_valid, X_test, y_train, y_valid, y_test = (
    processor.split_data(df)
)

pipeline = build_pipeline(
    "decision_tree",
    random_state=processor.random_state,
    parameters={},
)
param_grid = {
    "classifier__max_depth": [2, 4, 6],
    "classifier__min_samples_leaf": [1, 5, 10],
    "classifier__class_weight": [None, "balanced"],
}
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=processor.random_state,
)


## Поиск и независимая validation-оценка


In [ ]:
search = GridSearchCV(
    pipeline,
    param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1,
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best CV F1:", search.best_score_)
display(pd.Series(metrics(search.best_estimator_, X_valid, y_valid)))


Выбранные настройки переносим в training.json перед созданием выпуска. Этот ноутбук не меняет конфигурацию и роли моделей автоматически.
